# 🚀 HUẤN LUYỆN MODEL AI MINDMAP & FLASHCARD CHO DOCMIND (KAGGLE GPU T4)

> **Mục tiêu:** Fine-tune mô hình `Qwen2.5-3B-Instruct` bằng công nghệ **QLoRA (Unsloth)** trên 58 tài liệu bài giảng đã làm sạch.
>
> **Kết quả:** Mô hình có tư duy phân cấp thông minh, loại bỏ hoàn toàn rác slide/XML và tự động xuất ra file `.gguf` để nạp thẳng vào Ollama chạy cục bộ trên máy tính.

---

### ⚙️ YÊU CẦU CẤU HÌNH TRƯỚC KHI CHẠY:
Ở thanh menu bên phải màn hình (**Session Options**):
1. **Accelerator:** Chọn `GPU T4 x2` (hoặc `GPU T4`).
2. **Internet:** Gạt sang `ON` (bắt buộc để tải thư viện và base model).
3. Bấm nút **Add Input** $\rightarrow$ Chọn Dataset `mindmap-train` mà bạn đã tải lên.

## Bước 1: Cài đặt thư viện Unsloth (Tăng tốc huấn luyện x5 lần)

In [ ]:
# Cài đặt Unsloth và các thư viện cần thiết
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes
!pip install -q datasets requests python-docx

## Bước 2: Thiết lập môi trường làm việc & Sao chép dữ liệu

In [ ]:
import os, glob

# Tự động tìm thư mục dataset tải lên
input_dirs = glob.glob("/kaggle/input/*")
print("Các dataset tìm thấy:", input_dirs)

# Chép toàn bộ file vào thư mục làm việc
!cp -r /kaggle/input/*/* /kaggle/working/ 2>/dev/null || cp -r /kaggle/input/mindmap-train/* /kaggle/working/
%cd /kaggle/working
!ls -la

## Bước 3: Kiểm tra các tài liệu môn học (58 tài liệu JSON)

In [ ]:
import json, glob
doc_files = glob.glob("docs/*.json")
print(f"Tìm thấy {len(doc_files)} file tài liệu bài giảng.")
if doc_files:
    sample = json.load(open(doc_files[0], encoding='utf-8'))
    print("Mẫu file đầu tiên:", doc_files[0])
    print("Số sections:", len(sample.get("sections", [])))
    print("Độ dài extracted_text:", len(sample.get("extracted_text", "")), "ký tự")

## Bước 4: Tạo dữ liệu huấn luyện (Dataset Generation)

> **Lựa chọn A (Khuyên dùng - Chất lượng cao nhất):** Sử dụng Gemini 2.0 Flash làm Teacher để sinh nhãn có văn phong sư phạm chuẩn mực. Nhập API Key của bạn vào ô bên dưới.
>
> **Lựa chọn B (Miễn phí, Không cần API key):** Dùng bộ luật thông minh tích hợp sẵn trong `mindmap_engine.py` (chạy mất khoảng 30 giây).

In [ ]:
# === LỰA CHỌN A: Dùng Gemini Teacher (Khuyên dùng) ===
# Bỏ dấu # ở 2 dòng dưới và điền API key miễn phí từ https://aistudio.google.com
# os.environ["GEMINI_API_KEY"] = "AIzaSy..."
# !python make_training_data.py "docs/*.json" --teacher gemini --out train.jsonl

# === LỰA CHỌN B: Dùng Rule-Engine (Nhanh, Offline, Không cần API Key) ===
# Nếu không dùng Gemini, chạy lệnh dưới đây:
!python make_training_data.py "docs/*.json" --teacher engine --out train.jsonl

## Bước 5: Kiểm tra mẫu dữ liệu trước khi train

In [ ]:
import json, random
train_rows = [json.loads(line) for line in open("train.jsonl", encoding="utf-8")]
val_rows = [json.loads(line) for line in open("train.val.jsonl", encoding="utf-8")]
print(f"-> Tổng số mẫu huấn luyện (Train): {len(train_rows)}")
print(f"-> Tổng số mẫu kiểm thử (Validation): {len(val_rows)}")

# In thử 1 mẫu ngẫu nhiên
sample = random.choice(train_rows)
print("\n=== VĂN BẢN ĐẦU VÀO (INPUT) ===")
print(sample["messages"][1]["content"][:400] + "...")
print("\n=== KẾT QUẢ ĐẦU RA MẪU (JSON TARGET) ===")
print(sample["messages"][2]["content"][:500] + "...")

## Bước 6: Bắt đầu Huấn Luyện QLoRA (Khoảng 20 - 30 phút)

- **Base model:** `unsloth/Qwen2.5-3B-Instruct`
- **Phương pháp:** 4-bit QLoRA, rank=16, 7 target modules.
- **Đầu ra:** Tự động hợp nhất (merged 16-bit) và xuất cả định dạng `GGUF (q4_k_m)` cho Ollama.

In [ ]:
!python train_lora.py --train train.jsonl --val train.val.jsonl --base unsloth/Qwen2.5-3B-Instruct --epochs 2

## Bước 7: Đánh giá chất lượng mô hình sau khi train

In [ ]:
# Kiểm tra tỷ lệ trả về JSON hợp lệ trên tập validation
!python evaluate.py --model mindmap-model --val train.val.jsonl --n 30

## Bước 8: Đóng gói Model để tải về máy tính

Sau khi chạy ô này, hãy nhìn sang tab **Output** bên cột phải màn hình Kaggle $\rightarrow$ Bấm nút 3 chấm cạnh file `mindmap-model-v2.zip` $\rightarrow$ Chọn **Download**.

In [ ]:
!zip -rq mindmap-model-v2.zip mindmap-model mindmap-model-gguf train.jsonl train.val.jsonl
!ls -lh mindmap-model-v2.zip
print("\n=> HOÀN TẤT! File mindmap-model-v2.zip đã sẵn sàng ở tab Output bên phải để bạn tải về.")